# 04 — Teaching Meaning to Machines

This is the notebook where the rest of the series starts making sense. Take it slow.

Say a StoryVerse user searches for **"wizard school"**. Our library has a document about Hogwarts. The word "school" never appears in it. Neither does "wizard," probably. How is a computer supposed to know these two things are the same idea?

Plain word-matching can't do it — zero words overlap. We need a way to represent *meaning*, not just letters. That's what an **embedding** is.


## What an embedding actually is

An embedding is a list of numbers — a **vector** — that stands in for the meaning of a piece of text. Two pieces of text with similar meaning end up with similar numbers.

```
"king"   -> [0.2, 0.8, 0.1, 0.9, ...]   (hundreds of numbers)
"queen"  -> [0.2, 0.8, 0.1, 0.85, ...]  (very close to "king")
"table"  -> [0.9, 0.1, 0.8, 0.2, ...]   (nowhere close)
```

**Analogy:** GPS coordinates. Mumbai and Pune have coordinates that are close together. Mumbai and London don't. An embedding is GPS coordinates for *meaning* instead of *location* — except instead of 2 numbers (latitude, longitude), it's typically 384 to 1536 numbers. Each of those numbers is one **dimension** of "meaning space."

You can't picture 384 dimensions, and that's fine — the same math (distance, angle) that works in 2D still works in 384D, you just can't draw it on paper. Here's a toy 2D version to build intuition, using made-up coordinates:

```
"magic"     -> (0.90, 0.80)
"wizard"    -> (0.85, 0.82)
"sorcerer"  -> (0.88, 0.79)
"database"  -> (0.10, 0.20)
"SQL query" -> (0.12, 0.18)
```

Notice the magic-related words cluster near each other, and the tech words cluster near each other, far away from the first group. That clustering is the entire idea. Real embeddings just do this with far more dimensions and far more nuance.


## Measuring "close" — cosine similarity

We don't measure straight-line distance between two embeddings. We measure the **angle** between them. Why angle and not distance? Because direction carries the meaning; length mostly doesn't. Two arrows pointing the same way are "similar" no matter how long they are; an arrow pointing left and one pointing right are "different" even if they're the same length.

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{|A| \times |B|}$$

The result ranges from -1 (opposite meaning) to 1 (identical meaning). For real sentence embeddings, a rough feel: 0.7+ is genuinely related, below 0.4 is basically unrelated.


In [ ]:
import math


def dot_product(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b))


def magnitude(v: list[float]) -> float:
    return math.sqrt(sum(x**2 for x in v))


def cosine_similarity(a: list[float], b: list[float]) -> float:
    return dot_product(a, b) / (magnitude(a) * magnitude(b))


# Simple 3D vectors, chosen by hand to be obviously similar or different
king = [0.9, 0.1, 0.4]
queen = [0.85, 0.15, 0.42]
table = [0.1, 0.9, 0.2]

print("king vs queen:", round(cosine_similarity(king, queen), 3))
print("king vs table:", round(cosine_similarity(king, table), 3))


That's it. That handful of lines is the heart of every vector search system on the planet — from a 5-document toy project to a production system handling billions of vectors. Everything past this point is optimization on top of exactly this idea.


## Generating real embeddings

Time to stop hand-picking numbers and let a real model turn text into vectors. We'll use `sentence-transformers` with `all-MiniLM-L6-v2` — small, fast, good enough to learn on (production systems often reach for something bigger; see the comparison table near the end).


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "A young wizard attends a school of magic",
    "Harry Potter studies at Hogwarts",
    "A boy learns spells and potion-making",
    "A spacecraft travels through a wormhole",
    "An astronaut explores distant galaxies",
    "A soldier fights in a medieval war",
    "Kings and queens battle for a throne",
    "Database query optimization techniques",
    "SQL joins and indexing strategies",
    "Python programming for beginners",
]

embeddings = embedder.encode(sentences)
print("Shape:", embeddings.shape)  # (10 sentences, 384 numbers each)


Each sentence is now 384 numbers. None of them mean anything on their own — the meaning only shows up when you compare two sentences' numbers to each other.


## The big demo: keyword search vs. semantic search

Query: **"wizard school"**. Let's try it two ways.

**Keyword search** — does the sentence literally contain "wizard" or "school"?


In [ ]:
query = "wizard school"
query_words = set(query.lower().split())

print("Keyword matches:")
for s in sentences:
    overlap = query_words & set(s.lower().split())
    if overlap:
        print(f"  MATCH  ({overlap}) -- {s}")


Only sentence 1 literally contains those words. "Harry Potter studies at Hogwarts" — the single best answer to "wizard school" — doesn't match at all, because it never uses either word.

**Semantic search** — embed the query, then rank every sentence by cosine similarity.


In [ ]:
query_embedding = embedder.encode([query])[0]

ranked = sorted(
    zip(sentences, embeddings),
    key=lambda pair: cosine_similarity(query_embedding, pair[1]),
    reverse=True,
)

print(f"{'Rank':<5}{'Score':<8}Sentence")
for rank, (sentence, emb) in enumerate(ranked, start=1):
    score = cosine_similarity(query_embedding, emb)
    print(f"{rank:<5}{score:<8.2f}{sentence}")


"Harry Potter studies at Hogwarts" now ranks near the top, right alongside the other magic-themed sentences — even though it shares zero words with the query. The model isn't matching letters, it's matching meaning. The SQL and Python sentences fall to the bottom, exactly where they belong.

This is the entire reason embeddings replaced keyword search for this kind of problem.


## What the embedding space looks like

384 dimensions can't be drawn on a screen, but we can *project* them down to 2 dimensions (losing detail, keeping the overall shape) using **PCA**, just to see the clustering with our own eyes.


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
points_2d = pca.fit_transform(embeddings)

try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(7, 5))
    plt.scatter(points_2d[:, 0], points_2d[:, 1])
    for (x, y), label in zip(points_2d, sentences):
        plt.annotate(label[:24], (x, y), fontsize=8)
    plt.title("Sentence embeddings, projected to 2D")
    plt.show()
except ImportError:
    print("matplotlib not installed -- falling back to a plain coordinate printout:\n")
    for (x, y), label in zip(points_2d, sentences):
        print(f"  ({x:6.2f}, {y:6.2f})  {label}")


However you view it, the same pattern shows up: the magic sentences cluster together, the space sentences cluster together, the tech sentences cluster together, and none of the clusters overlap. The model learned this structure just from reading enormous amounts of text — nobody hand-labeled "these are the fantasy sentences."


## Saving the embeddings for real use

Let's embed our actual 5 StoryVerse stories from notebook 02 and save the result. No vector database yet — just a JSON file. This is, genuinely, the most primitive vector store there is.


In [ ]:
import os
import json

DATA_DIR = os.path.join("..", "data")
STORIES_DIR = os.path.join(DATA_DIR, "stories")


def load_documents(folder_path: str) -> list[dict]:
    docs = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename)) as f:
                docs.append({"filename": filename, "content": f.read()})
    return docs


story_docs = load_documents(STORIES_DIR)
story_embeddings = embedder.encode([d["content"] for d in story_docs])

embedded_docs = [
    {"filename": d["filename"], "content": d["content"], "embedding": emb.tolist()}
    for d, emb in zip(story_docs, story_embeddings)
]

embeddings_path = os.path.join(DATA_DIR, "embeddings.json")
with open(embeddings_path, "w") as f:
    json.dump(embedded_docs, f)

print(f"Saved {len(embedded_docs)} embedded documents to {embeddings_path}")
print("Each embedding has", len(embedded_docs[0]["embedding"]), "dimensions")


## Which embedding model should you actually use?

| Model | Dimensions | Speed | Quality | Best for |
|---|---|---|---|---|
| all-MiniLM-L6-v2 | 384 | Fast | Good | Prototyping (what we used today) |
| all-mpnet-base-v2 | 768 | Medium | Better | Production baseline |
| BGE-large | 1024 | Slow | Best (open) | High-accuracy retrieval |
| OpenAI text-embedding-3-small | 1536 | API call | Excellent | Cloud-hosted apps |

Start with MiniLM. Upgrade only once you've actually measured that retrieval quality is the bottleneck — notebook 10 shows you how to measure that instead of guessing.


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Embedding | A list of numbers standing in for a piece of text's meaning |
| Vector | Another word for that list of numbers |
| Dimension | One number in the vector; embeddings typically have hundreds |
| Semantic space | The imaginary space where similar-meaning text ends up close together |
| Cosine similarity | A score (-1 to 1) for how closely two vectors point in the same direction |
| Dense embedding | An embedding where nearly every number is non-zero (as opposed to sparse keyword vectors) |

Text can now be compared by *meaning*, not just by *letters*. We have embeddings for our 5 stories saved to disk. Next notebook: build a real search engine on top of them.

**Exercises:**

- Embed "Baahubali" and "an epic war drama about reclaiming a throne" — what's the cosine similarity?
- Embed your own name and a random unrelated word — what score do you get?
- Find two sentences that *sound* completely different but that you'd expect to be semantically close — check if the model agrees with you.
